## Download Dataset

In [1]:
import pathlib
import requests

# Create data directory if it doesn't exist
pathlib.Path("data").mkdir(parents=True, exist_ok=True)

# Download the file
prefix = "https://raw.githubusercontent.com/juand-r/entity-recognition-datasets/refs/heads/master/data/BTC/CONLL-format/data/"
filenames = [f"{name}.conll" for name in ["a", "b", "e", "f", "g", "h"]]
for filename in filenames:
    url = f"{prefix}{filename}"
    response = requests.get(url)
    if response.status_code == 200:
        with open(f"data/{filename}", "wb") as file:
            file.write(response.content)
        print(f"Downloaded {filename}")
    else:
        print(f"Failed to download {filename} from {url}")

Downloaded a.conll
Downloaded b.conll
Downloaded e.conll
Downloaded f.conll
Downloaded g.conll
Downloaded h.conll


## Parse Data

In [2]:
# Process the downloaded files
results = []
for filename in filenames:
    path = pathlib.Path(f"data/{filename}")
    with open(path, "r", encoding="utf-8") as file:
        content = file.read().strip()
        sentences = content.split("\n\n")
        for sentence in sentences:
            pairs = sentence.split("\n")
            data = [tuple(pair.split("\t")) for pair in pairs]
            results.append(data)
print(f"Processed {len(results)} sentences from {len(filenames)} files.")

Processed 9339 sentences from 6 files.


## Encode the labels into integers

In [3]:
# Create a mapping for labels
labels = ["O", "B-LOC", "I-LOC", "B-ORG", "I-ORG", "B-PER", "I-PER"]
mapping = {label: i for i, label in enumerate(labels)}

# Apply the mapping to the results
for i, sentence in enumerate(results):
    for j, (word, label) in enumerate(sentence):
        results[i][j] = (word, mapping[label])
print("Label mapping applied to all sentences.")

Label mapping applied to all sentences.


## Split the results into train, validation, and test sets

In [4]:
from sklearn.model_selection import train_test_split

# Split the data into test, validation, and training sets in a 80/10/10 ratio
train_data, test_data = train_test_split(results, test_size=0.2)
val_data, test_data = train_test_split(test_data, test_size=0.5)
print(f"Training data: {len(train_data)} sentences")
print(f"Validation data: {len(val_data)} sentences")
print(f"Test data: {len(test_data)} sentences")

Training data: 7471 sentences
Validation data: 934 sentences
Test data: 934 sentences


## Prepare the datasets

In [5]:
from datasets import Dataset, DatasetDict


# Create datasets
def create_dataset(data):
    buffer = []
    for sentence in data:
        tokens, labels = zip(*sentence)
        tokens, labels = list(tokens), list(labels)
        buffer.append({"tokens": tokens, "labels": labels})
    return Dataset.from_list(buffer)


# Create the dataset dictionary
dataset = DatasetDict(
    {
        "train": create_dataset(train_data),
        "validation": create_dataset(val_data),
        "test": create_dataset(test_data),
    }
)
print("Dataset created with train, validation, and test splits.")

c:\Users\0713E\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset created with train, validation, and test splits.


## Tokenize the text

In [6]:
from transformers import AutoTokenizer

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased", is_split_into_words=True
)


# Tokenize the dataset and align labels
def tokenize_and_align_labels(data):
    tokenized_inputs = tokenizer(
        data["tokens"], truncation=True, is_split_into_words=True
    )
    labels = [-100] * len(tokenized_inputs.input_ids)
    word_idxs = tokenized_inputs.word_ids()
    for index, word_idx in enumerate(word_idxs):
        if word_idx is None:
            continue
        labels[index] = data["labels"][word_idx]
    return {
        "input_ids": tokenized_inputs.input_ids,
        "attention_mask": tokenized_inputs.attention_mask,
        "labels": labels,
    }

# Apply the tokenization and alignment to the dataset
dataset = dataset.map(
    tokenize_and_align_labels,
    remove_columns=["tokens", "labels"],
    desc="Tokenizing and aligning labels",
)
print("Tokenization and label alignment completed.")

Tokenizing and aligning labels: 100%|██████████| 934/934 [00:00<00:00, 3519.24 examples/s]

Tokenization and label alignment completed.


## Fine-tune the model

In [7]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import evaluate
import torch

# Load the model
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=len(labels)
).to("cuda" if torch.cuda.is_available() else "cpu")

# Define training arguments
training_args = TrainingArguments(
    output_dir="./fine-tuned-model",
    eval_strategy="epoch",
    num_train_epochs=3,
    weight_decay=0.005,
)

# Define the data collator
data_collator = DataCollatorForTokenClassification(tokenizer)


# Define the metric for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)
    id2label = {i: label for label, i in mapping.items()}
    true_predictions = [
        [id2label[pred] for pred, label in zip(pred_seq, label_seq) if label != -100]
        for pred_seq, label_seq in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[label] for pred, label in zip(pred_seq, label_seq) if label != -100]
        for pred_seq, label_seq in zip(predictions, labels)
    ]
    metric = evaluate.load("seqeval")
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

# Train the model
trainer.train()
print("Training completed.")

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.229000,0.157686,0.745374,0.731584,0.738415,0.946985
2,0.108900,0.155208,0.775585,0.780020,0.777796,0.953989
3,0.065800,0.177710,0.765579,0.781029,0.773227,0.952815


Training completed.


## Evaluate the model

In [13]:
# Evaluate the model
eval_results = trainer.evaluate(dataset["test"])
print("Evaluation results:")
print(f"Precision: {eval_results['eval_precision']:.4f}")
print(f"Recall: {eval_results['eval_recall']:.4f}")
print(f"F1 Score: {eval_results['eval_f1']:.4f}")
print(f"Accuracy: {eval_results['eval_accuracy']:.4f}")

Evaluation results:
Precision: 0.7627
Recall: 0.7534
F1 Score: 0.7580
Accuracy: 0.9514


## Test the model

In [14]:
from transformers import pipeline

# Give some example sentences for Named Entity Recognition (NER)
sentences = [
    "The capital of France is Paris.",
    "Apple Inc. is a technology company based in Cupertino.",
    "Barack Obama was the 44th President of the United States.",
]

# Create a Named Entity Recognition pipeline
ner = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
)

# Run NER on the example sentences
group2label = {f"LABEL_{i}": label for label, i in mapping.items()}
for sentence in sentences:
    results = ner(sentence)
    print(f"Sentence: {sentence}")
    for entity in results:
        print(
            f"Entity: {entity['word']}, Label: {group2label[entity['entity_group']]}, Score: {entity['score']:.4f}"
        )
    print()

Device set to use cuda:0


Sentence: The capital of France is Paris.
Entity: the capital of, Label: O, Score: 0.9752
Entity: france, Label: B-LOC, Score: 0.9913
Entity: is, Label: O, Score: 0.9892
Entity: paris, Label: B-LOC, Score: 0.9948
Entity: ., Label: O, Score: 0.9109

Sentence: Apple Inc. is a technology company based in Cupertino.
Entity: apple, Label: B-ORG, Score: 0.9828
Entity: inc., Label: I-ORG, Score: 0.9587
Entity: is a technology company based in, Label: O, Score: 0.9955
Entity: cupertino, Label: B-LOC, Score: 0.9948
Entity: ., Label: O, Score: 0.4668

Sentence: Barack Obama was the 44th President of the United States.
Entity: barack, Label: B-PER, Score: 0.9866
Entity: obama, Label: I-PER, Score: 0.9965
Entity: was the 44th president of the, Label: O, Score: 0.9949
Entity: united, Label: B-LOC, Score: 0.8989
Entity: states, Label: I-LOC, Score: 0.8235
Entity: ., Label: O, Score: 0.9932

